# KLE under CoT — AUROC comparison charts

Step-by-step chart notebook. **N=1000, seed=42, LLM-as-Judge labels.**
Uses the AUROC values stored in the result files (no recomputation, no CIs —
bootstrapped versions will live in a separate chart).

- **Chart 1:** AUROC by condition, black-box vs white-box panels, all three
  datasets, Llama-3.1-8B only.
- **Chart 2:** MoreHopQA only — Llama-3.1-8B vs Mistral-7B, black-box and
  white-box panels.
- **Chart 3:** Chart 1 with 95% bootstrap confidence intervals (question-level
  resampling, B=2000).
- **Chart 4:** C2 (holistic CoT) vs C4 (stepwise) head-to-head, with bootstrap
  CIs, black-box and white-box panels.
- **Chart 5:** MoreHopQA — C2 vs C4 across models (Llama-3.1-8B vs Mistral-7B),
  with bootstrap CIs.
- **Charts 6–7:** C2 vs C4 plus external baselines (white-box: Semantic Entropy,
  LLM-Check; black-box: SelfCheckGPT-NLI), Llama-3.1-8B, all three datasets —
  Chart 6 as plain bars, Chart 7 with bootstrap CIs.
- **Charts 8–9:** AUROC and hallucination rate side by side per condition and
  dataset (Llama-3.1-8B); Chart 8 white-box, Chart 9 black-box.
- **Charts 10–11:** token cost — Chart 10: AUROC vs mean tokens/question per
  condition (black-box and white-box); Chart 11: performance-cost trade-off
  scatter (adapted from the TriviaQA analysis notebook, time axis replaced by
  token counts), one row per box type.
- **Chart 12:** Chart 10 with the token row stacked into generation vs
  forward-pass components.
- **Charts 13–18:** additional comparison metrics (AUPRC, PRR, FPR@95TPR), one
  figure per metric — Charts 13–15 for C1–C4, Charts 16–18 for C2/C4 vs the
  external baselines. All with bootstrap CIs.

Run from the `results/` directory. Figures are saved as PNG under `results/figures/`.


In [ ]:
import json, os
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.grid": True,
                     "grid.alpha": 0.3, "axes.axisbelow": True})

DATA_DIR = "."            # run from the results/ directory
FIG_DIR = "./figures"
os.makedirs(FIG_DIR, exist_ok=True)

DATASETS = ["triviaqa", "nqopen", "morehopqa"]
DS_LABEL = {"triviaqa": "TriviaQA", "nqopen": "NQ-Open", "morehopqa": "MoreHopQA"}

# Fixed condition colors (consistent with the full-analysis notebook)
COL = {"C1": "#2a78d6", "C2": "#1baf7a", "C3": "#eda100", "C4": "#4a3aa7"}
CHANCE = "#9a9992"

# (dataset, model, box) -> {cond: file path}
REG = {}
REG[("triviaqa", "8b", "bb")] = {
    "C1": "black-box/triviaqa/c1_triviaqa_llama_bb.json",
    "C2": "black-box/triviaqa/c2_triviaqa_llama_bb.json",
    "C3": "black-box/triviaqa/c3_triviaiqa_llama_bb.json",  # typo in file name
    "C4": "black-box/triviaqa/c4_triviaqa_llama_bb.json"}
REG[("triviaqa", "8b", "wb")] = {
    f"C{i}": f"white-box/triviaqa/c{i}_triviaqa_llama_wb.json" for i in range(1, 5)}
for ds in ["nqopen", "morehopqa"]:
    for model in (["8b"] if ds == "nqopen" else ["8b", "mistral"]):
        REG[(ds, model, "bb")] = {
            f"C{i}": f"black-box/{ds}/c{i}_{ds}_{model}_BB_results.json" for i in range(1, 5)}
        REG[(ds, model, "wb")] = {
            f"C{i}": f"white-box/{ds}/c{i}_{ds}_{model}_WB_results.json" for i in range(1, 5)}

# AUROC[(dataset, model, box, cond)] -> float
# C4 expands into its aggregation variants: "C4:max", "C4:mean", "C4:attn_mean"
# HRATE[(dataset, model, box, cond)] -> hallucination rate (one per file, so C4
# has a single rate shared by its aggregation variants)
AUROC, HRATE = {}, {}
for (ds, model, box), files in REG.items():
    for cond, rel in files.items():
        with open(os.path.join(DATA_DIR, rel)) as f:
            d = json.load(f)
        if cond == "C4":
            for agg, val in d["auroc"].items():
                AUROC[(ds, model, box, f"C4:{agg}")] = val
        else:
            AUROC[(ds, model, box, cond)] = d["auroc"]
        HRATE[(ds, model, box, cond)] = d["hallucination_rate"]

print(f"Loaded {len(AUROC)} AUROC values, {len(HRATE)} hallucination rates")

## Chart 1 — AUROC by condition (Llama-3.1-8B)

Grouped bars per dataset; separate panels for black-box and white-box.
All C4 aggregation variants are shown: `max`/`mean` in black-box (attention
aggregation is removed there by design), `max`/`mean`/`attn_mean` in white-box.
C4 variants share the C4 color and are distinguished by hatching.

In [ ]:
BARS = {
    "bb": [("C1", "C1", ""), ("C2", "C2", ""), ("C3", "C3", ""),
           ("C4:max", "C4 (max)", ""), ("C4:mean", "C4 (mean)", "//")],
    "wb": [("C1", "C1", ""), ("C2", "C2", ""), ("C3", "C3", ""),
           ("C4:max", "C4 (max)", ""), ("C4:mean", "C4 (mean)", "//"),
           ("C4:attn_mean", "C4 (attn_mean)", "xx")],
}

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8), sharey=True)
for ax, box in zip(axes, ["bb", "wb"]):
    bars = BARS[box]
    k = len(bars)
    w = 0.8 / k
    for j, (cond, label, hatch) in enumerate(bars):
        xs = [i + (j - (k - 1) / 2) * w for i in range(len(DATASETS))]
        ys = [AUROC[(ds, "8b", box, cond)] for ds in DATASETS]
        ax.bar(xs, ys, width=w * 0.92, color=COL[cond.split(":")[0]],
               edgecolor="black", linewidth=0.5, hatch=hatch, label=label)
        for x, v in zip(xs, ys):
            ax.annotate(f"{v:.2f}", (x, v), textcoords="offset points",
                        xytext=(0, 2), ha="center", fontsize=7)
    ax.axhline(0.5, color=CHANCE, ls="--", lw=1, label="chance")
    ax.set_xticks(range(len(DATASETS)))
    ax.set_xticklabels([DS_LABEL[d] for d in DATASETS])
    ax.set_ylim(0, 1.0)
    ax.set_title("Black-box" if box == "bb" else "White-box")
axes[0].set_ylabel("AUROC")
axes[1].legend(loc="upper right", fontsize=8, ncol=2)
fig.suptitle("Hallucination detection AUROC by condition (Llama-3.1-8B, N=1000)", y=1.02)
plt.tight_layout()
fig.savefig(f"{FIG_DIR}/cmp1_auroc_by_condition_8b.png", bbox_inches="tight")
plt.show()

**Notes — Chart 1**

- Reading: AUROC = probability that a random hallucinated answer gets a higher
  uncertainty score than a random correct one. 1.0 = perfect detector, 0.5 =
  chance (dashed line), below 0.5 = the signal is *inverted*.
- **Single-hop (TriviaQA, NQ-Open):** C3 (CoT + answer-span extraction) is the
  best black-box detector (0.86 / 0.74), slightly ahead of the C1 direct-answer
  baseline (0.84 / 0.73). CoT helps, but only if the final answer span is
  isolated before clustering.
- **C2 is consistently the weakest KLE-CoT variant** on single-hop: clustering
  the whole chain inflates semantic diversity from reasoning text and dilutes
  the uncertainty signal. Expected by design, not a bug.
- **Multi-hop (MoreHopQA) flips the picture:** C1 collapses below chance (0.37)
  — the model gives *consistently wrong* direct answers, so low entropy no
  longer means correct. C4 (stepwise, `mean`) becomes the best black-box
  detector (0.71).
- **C4 aggregation is dataset-dependent:** `max` wins on single-hop black-box
  (TriviaQA 0.68 vs 0.63), `mean` wins on multi-hop (MoreHopQA 0.71 vs 0.59).
  Worth an explicit discussion point rather than silently picking one.
- **Black-box vs white-box:** for C1/C3 on single-hop, black-box KLE *beats* the
  white-box hidden-state variant (e.g. TriviaQA C3: 0.86 vs 0.82) — semantic
  clustering over text is the stronger signal there. In white-box, C4 with
  attention aggregation is competitive (TriviaQA 0.79) and best on MoreHopQA.

## Chart 2 — MoreHopQA: Llama-3.1-8B vs Mistral-7B

Same layout, but the x-axis lists the conditions (with C4 variants) and color
encodes the model.

In [ ]:
MODEL_COL = {"8b": "#2a78d6", "mistral": "#eb6834"}
MODEL_LABEL = {"8b": "Llama-3.1-8B", "mistral": "Mistral-7B"}
CONDS = {
    "bb": ["C1", "C2", "C3", "C4:max", "C4:mean"],
    "wb": ["C1", "C2", "C3", "C4:max", "C4:mean", "C4:attn_mean"],
}

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8), sharey=True)
for ax, box in zip(axes, ["bb", "wb"]):
    conds = CONDS[box]
    w = 0.38
    for j, model in enumerate(["8b", "mistral"]):
        xs = [i + (j - 0.5) * w for i in range(len(conds))]
        ys = [AUROC[("morehopqa", model, box, c)] for c in conds]
        ax.bar(xs, ys, width=w * 0.92, color=MODEL_COL[model],
               edgecolor="black", linewidth=0.5, label=MODEL_LABEL[model])
        for x, v in zip(xs, ys):
            ax.annotate(f"{v:.2f}", (x, v), textcoords="offset points",
                        xytext=(0, 2), ha="center", fontsize=7)
    ax.axhline(0.5, color=CHANCE, ls="--", lw=1, label="chance")
    ax.set_xticks(range(len(conds)))
    ax.set_xticklabels([c.replace(":", "\n") for c in conds], fontsize=9)
    ax.set_ylim(0, 1.0)
    ax.set_title("Black-box" if box == "bb" else "White-box")
axes[0].set_ylabel("AUROC")
axes[1].legend(loc="upper right", fontsize=9)
fig.suptitle("MoreHopQA — AUROC by condition and model (N=1000)", y=1.02)
plt.tight_layout()
fig.savefig(f"{FIG_DIR}/cmp2_morehopqa_llama_vs_mistral.png", bbox_inches="tight")
plt.show()

**Notes — Chart 2**

- Purpose: check whether the MoreHopQA findings are model-specific or replicate
  across generators. Both models were run with identical configs (N=1000,
  10 samples, seed=42).
- **C1 is below or near chance for both models** (Llama 0.37, Mistral 0.46):
  the "confidently wrong on multi-hop" failure of the direct-answer baseline is
  not a Llama artifact.
- **C3 is remarkably model-stable in black-box** (0.681 vs 0.684) — answer-span
  extraction transfers across generators almost unchanged.
- **The C4 stepwise advantage is model-dependent:** for Llama, C4 `mean` (0.71)
  is the best black-box detector; for Mistral, C3 (0.68) stays ahead of C4
  `mean` (0.61). Honest caveat for RQ1: step-level uncertainty helps on
  multi-hop, but the size of the gain varies by model.
- White-box panel: Mistral's C1 (0.66) is unusually strong compared to Llama's
  (0.47), while its C2/C3 are weaker (~0.51–0.55). Hidden-state signals behave
  less consistently across models than the black-box semantic ones.
- Context: hallucination rates on MoreHopQA are 63–89% for both models — a very
  hard task; labels are heavily imbalanced (AUROC tolerates this, but accuracy-
  style metrics would not).

## Chart 3 — bootstrap view: AUROC point estimates with 95% CIs

Same comparison as Chart 1, but drawn as **dot-and-whisker**: the dot is the
AUROC point estimate, the whisker the 95% CI from a nonparametric bootstrap
(questions resampled with replacement, B=2000, recomputed from the per-question
scores in `details`). Without bar bodies the y-axis can zoom into the data
range, so interval widths and overlaps are actually readable. C4 variants share
the C4 color and are distinguished by marker shape (●=`max`, ■=`mean`,
▲=`attn_mean`). The helper cell below takes ~1 min on first run.

In [ ]:
import numpy as np
from scipy.stats import rankdata

N_BOOT, RNG_SEED = 2000, 0

def extract_scores(d, box, cond):
    """Per-question uncertainty scores + hallucination labels for one run."""
    if cond.startswith("C4"):
        agg = cond.split(":")[1]
        det = [r for r in d["details"] if r.get("error") is None and r.get("agg")]
        s = np.array([r["agg"][agg] for r in det], float)
        y = np.array([int(r["is_halluc"]) for r in det], int)
    else:
        det = d["details"]
        s = np.array([r["kle_full" if box == "bb" else "vne"] for r in det], float)
        y = np.array([0 if r["judge_correct"] else 1 for r in det], int)
    return s, y

def fast_auroc(y, s):
    """Rank-based AUROC (equivalent to sklearn's roc_auc_score)."""
    r = rankdata(s)
    n1 = int(y.sum()); n0 = len(y) - n1
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def bootstrap_ci(y, s, n_boot=N_BOOT, seed=RNG_SEED):
    rng = np.random.default_rng(seed)
    n = len(y); out = np.full(n_boot, np.nan)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        yb = y[idx]
        if 0 < yb.sum() < n:
            out[b] = fast_auroc(yb, s[idx])
    return np.nanpercentile(out, [2.5, 97.5])

# CI[(ds, box, cond)] = (auroc, lo, hi) for every bar in Chart 1
CI = {}
for ds in DATASETS:
    for box in ["bb", "wb"]:
        cache = {}
        for cond, _, _ in BARS[box]:
            base = cond.split(":")[0]
            if base not in cache:
                with open(os.path.join(DATA_DIR, REG[(ds, "8b", box)][base])) as f:
                    cache[base] = json.load(f)
            s, y = extract_scores(cache[base], box, cond)
            lo, hi = bootstrap_ci(y, s)
            CI[(ds, box, cond)] = (fast_auroc(y, s), lo, hi)
print(f"Computed {len(CI)} bootstrap CIs")

In [ ]:
MARKER = {"C1": "o", "C2": "o", "C3": "o",
          "C4:max": "o", "C4:mean": "s", "C4:attn_mean": "^"}

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8), sharey=True)
for ax, box in zip(axes, ["bb", "wb"]):
    series = BARS[box]
    k = len(series)
    w = 0.8 / k
    for j, (cond, label, _) in enumerate(series):
        xs = [i + (j - (k - 1) / 2) * w for i in range(len(DATASETS))]
        vals = [CI[(ds, box, cond)] for ds in DATASETS]
        ys = [v[0] for v in vals]
        yerr = [[v[0] - v[1] for v in vals], [v[2] - v[0] for v in vals]]
        ax.errorbar(xs, ys, yerr=yerr, fmt=MARKER[cond], ms=6, capsize=3,
                    lw=1.5, ls="none", color=COL[cond.split(":")[0]], label=label)
        for x, v in zip(xs, ys):
            ax.annotate(f"{v:.2f}", (x, v), textcoords="offset points",
                        xytext=(6, -3), ha="left", fontsize=6.5)
    ax.axhline(0.5, color=CHANCE, ls="--", lw=1, label="chance")
    ax.set_xticks(range(len(DATASETS)))
    ax.set_xticklabels([DS_LABEL[d] for d in DATASETS])
    ax.set_title("Black-box" if box == "bb" else "White-box")
axes[0].set_ylabel("AUROC (95% bootstrap CI)")
axes[1].legend(loc="lower left", fontsize=8, ncol=2)
fig.suptitle("Hallucination detection AUROC by condition, with bootstrap CIs "
             "(Llama-3.1-8B, N=1000)", y=1.02)
plt.tight_layout()
fig.savefig(f"{FIG_DIR}/cmp3_auroc_by_condition_8b_ci.png", bbox_inches="tight")
plt.show()

**Notes — Chart 3**

- The whiskers are 95% percentile intervals from resampling *questions* with
  replacement (B=2000). They quantify how much AUROC would wobble if we had
  drawn a different N=1000 question sample — nothing more.
- Typical CI width at N=1000 is roughly ±0.02–0.05; anything narrower than that
  gap between two bars should not be over-interpreted.
- **C1 on MoreHopQA (black-box): the whole CI sits below 0.5** — the inverted
  signal is statistically solid, not sampling noise.
- **Large gaps are clearly real:** e.g. MoreHopQA black-box C4 `mean` vs C1
  (0.71 vs 0.37) — the intervals are far apart.
- **Small gaps need care:** TriviaQA black-box C3 vs C1 (0.86 vs 0.84) has
  overlapping CIs. Overlap does *not* prove "no difference": these bars share
  the same questions, so the honest test is a *paired* bootstrap on the AUROC
  difference (available in the full-analysis notebook, section 13; C3 - C1 is
  significant there).
- These CIs capture question-sampling uncertainty only; generation randomness
  (temperature, seed) is a separate source — all runs use a single seed (42).

## Chart 4 — C2 (holistic CoT) vs C4 (stepwise) with 95% bootstrap CIs

Head-to-head of the two CoT granularities, Llama-3.1-8B: C2 treats the whole
chain as one unit, C4 scores it step by step. Dot-and-whisker as in Chart 3;
requires the CI cache computed by the Chart 3 helper cell. C4 markers:
●=`max`, ■=`mean`, ▲=`attn_mean`.

In [ ]:
C2C4 = {
    "bb": [("C2", "C2 (holistic)", "o"),
           ("C4:max", "C4 (max)", "o"), ("C4:mean", "C4 (mean)", "s")],
    "wb": [("C2", "C2 (holistic)", "o"),
           ("C4:max", "C4 (max)", "o"), ("C4:mean", "C4 (mean)", "s"),
           ("C4:attn_mean", "C4 (attn_mean)", "^")],
}

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8), sharey=True)
for ax, box in zip(axes, ["bb", "wb"]):
    series = C2C4[box]
    k = len(series)
    w = 0.7 / k
    for j, (cond, label, mk) in enumerate(series):
        xs = [i + (j - (k - 1) / 2) * w for i in range(len(DATASETS))]
        vals = [CI[(ds, box, cond)] for ds in DATASETS]
        ys = [v[0] for v in vals]
        yerr = [[v[0] - v[1] for v in vals], [v[2] - v[0] for v in vals]]
        ax.errorbar(xs, ys, yerr=yerr, fmt=mk, ms=6.5, capsize=3,
                    lw=1.5, ls="none", color=COL[cond.split(":")[0]], label=label)
        for x, v in zip(xs, ys):
            ax.annotate(f"{v:.2f}", (x, v), textcoords="offset points",
                        xytext=(6, -3), ha="left", fontsize=7)
    ax.axhline(0.5, color=CHANCE, ls="--", lw=1, label="chance")
    ax.set_xticks(range(len(DATASETS)))
    ax.set_xticklabels([DS_LABEL[d] for d in DATASETS])
    ax.set_title("Black-box" if box == "bb" else "White-box")
axes[0].set_ylabel("AUROC (95% bootstrap CI)")
axes[1].legend(loc="lower left", fontsize=8)
fig.suptitle("Holistic CoT (C2) vs stepwise (C4) — AUROC with bootstrap CIs "
             "(Llama-3.1-8B, N=1000)", y=1.02)
plt.tight_layout()
fig.savefig(f"{FIG_DIR}/cmp4_c2_vs_c4_ci.png", bbox_inches="tight")
plt.show()

**Notes — Chart 4**

- This chart isolates the *granularity* question of RQ1: given the same CoT
  prompting, does scoring the chain **step by step (C4)** beat scoring it **as
  one unit (C2)**? Both use their own generations; the comparison is
  question-paired at the dataset level.
- **White-box: stepwise wins everywhere.** C4 (`attn_mean`/`mean`) is above C2
  on all three datasets — TriviaQA 0.79 vs 0.69, NQ-Open 0.72 vs 0.67,
  MoreHopQA 0.68 vs 0.64 — and on TriviaQA the CIs are clearly separated.
  Step-level decomposition consistently adds signal in hidden-state space.
- **Black-box: stepwise wins only on multi-hop.** On MoreHopQA, C4 `mean`
  (0.71) is far above C2 (0.55) with non-overlapping CIs. On TriviaQA the
  ordering flips (C2 0.71 vs C4 `max` 0.68), and on NQ-Open they are on par
  (~0.64) — single-hop chains have too few informative steps for the black-box
  step signal.
- C2's weakness is flat across datasets (0.55–0.71): holistic clustering is
  diluted by surface diversity of the reasoning text regardless of hop count.
- Combined takeaway for the meeting: *step-level granularity is the right
  choice whenever the signal source is rich (white-box) or the task is
  multi-hop (black-box); it is not a free win on single-hop black-box.*
- Overlapping-CI pairs (e.g. NQ-Open black-box) should be settled with the
  paired bootstrap (full-analysis notebook, section 13), not by eyeballing
  interval overlap.

## Chart 5 — MoreHopQA: C2 vs C4 across models, with 95% bootstrap CIs

Model robustness check for the granularity comparison of Chart 4, on the
multi-hop dataset only. Color encodes the model (as in Chart 2); the x-axis
lists the C2/C4 variants. Llama CIs are reused from the Chart 3 cache; the
Mistral CIs are computed below (a few seconds).

In [ ]:
# Extend the CI cache with Mistral / MoreHopQA runs
CI5 = {}
for model in ["8b", "mistral"]:
    for box in ["bb", "wb"]:
        cache = {}
        for cond, _, _ in C2C4[box]:
            if model == "8b":
                CI5[(model, box, cond)] = CI[("morehopqa", box, cond)]
                continue
            base = cond.split(":")[0]
            if base not in cache:
                with open(os.path.join(DATA_DIR, REG[("morehopqa", model, box)][base])) as f:
                    cache[base] = json.load(f)
            s, y = extract_scores(cache[base], box, cond)
            lo, hi = bootstrap_ci(y, s)
            CI5[(model, box, cond)] = (fast_auroc(y, s), lo, hi)
print(f"CI cache for chart 5: {len(CI5)} entries")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8), sharey=True)
for ax, box in zip(axes, ["bb", "wb"]):
    conds = [c for c, _, _ in C2C4[box]]
    labels = [lab for _, lab, _ in C2C4[box]]
    w = 0.3
    for j, model in enumerate(["8b", "mistral"]):
        xs = [i + (j - 0.5) * w for i in range(len(conds))]
        vals = [CI5[(model, box, c)] for c in conds]
        ys = [v[0] for v in vals]
        yerr = [[v[0] - v[1] for v in vals], [v[2] - v[0] for v in vals]]
        ax.errorbar(xs, ys, yerr=yerr, fmt="o", ms=6.5, capsize=3,
                    lw=1.5, ls="none", color=MODEL_COL[model],
                    label=MODEL_LABEL[model])
        for x, v in zip(xs, ys):
            ax.annotate(f"{v:.2f}", (x, v), textcoords="offset points",
                        xytext=(6, -3), ha="left", fontsize=7)
    ax.axhline(0.5, color=CHANCE, ls="--", lw=1, label="chance")
    ax.set_xticks(range(len(conds)))
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_title("Black-box" if box == "bb" else "White-box")
axes[0].set_ylabel("AUROC (95% bootstrap CI)")
axes[1].legend(loc="lower right", fontsize=9)
fig.suptitle("MoreHopQA — C2 vs C4 by model, with bootstrap CIs (N=1000)", y=1.02)
plt.tight_layout()
fig.savefig(f"{FIG_DIR}/cmp5_morehopqa_c2_vs_c4_models_ci.png", bbox_inches="tight")
plt.show()

**Notes — Chart 5**

- Question answered here: *does the multi-hop stepwise advantage (Chart 4)
  survive a change of generator model?* Both models: identical configs, N=1000,
  seed=42.
- **C2 black-box is model-invariant and weak** (Llama 0.55, Mistral 0.55, CIs
  fully overlapping): holistic-chain dilution is a property of the method, not
  of the model.
- **C4 > C2 holds for both models in both boxes**, but the effect size differs:
  black-box `mean` gains +0.17 for Llama (0.71 vs 0.55, CIs clearly separated)
  and only +0.07 for Mistral (0.61 vs 0.55, CIs touch). White-box: Mistral
  +0.10 (0.61 vs 0.51), Llama +0.04 (0.68 vs 0.64, overlapping CIs).
- **Best aggregation depends on the model too:** Llama's black-box winner is
  clearly `mean` (0.71 vs `max` 0.59); for Mistral the two variants are within
  noise of each other (0.58–0.61).
- Mistral's white-box variants are all ≈0.61 regardless of aggregation —
  attention weighting adds nothing there.
- Meeting takeaway: the *direction* (stepwise beats holistic on multi-hop) is
  model-robust; the *magnitude* is model-dependent, strongest for Llama
  black-box. Borderline pairs should again go through the paired bootstrap
  before being called significant.

## Charts 6–7 — C2 vs C4 against external baselines (Llama-3.1-8B)

Adds the external baselines to the C2-vs-C4 comparison, per box type:

- **White-box panel:** Semantic Entropy and LLM-Check (its best detector,
  `logit_entropy`). Semantic Entropy is computed post-hoc from the per-cluster
  log-likelihoods (`log_lik_per_sem_id`, already sum-normalized) stored in the
  C2 black-box logs — the standard Rao-Blackwellized estimator. It needs token
  likelihoods, so it counts as white-box.
- **Black-box panel:** SelfCheckGPT-NLI.

All baselines operate on C2 (holistic CoT) outputs; C4 uses its own stepwise
generations. Chart 6 = plain bars, Chart 7 = the bootstrap (dot-and-whisker)
version. The helper below needs the Chart 3 helpers (`fast_auroc`,
`bootstrap_ci`), so run the notebook top to bottom.

In [ ]:
from scipy.special import logsumexp

BCOL = {"SelfCheckGPT": "#e87ba4", "LLM-Check": "#eb6834", "SemanticEntropy": "#008300"}

def semantic_entropy(log_lik_per_sem_id):
    """Rao-Blackwellized semantic entropy from per-cluster log-likelihoods."""
    ll = np.asarray(log_lik_per_sem_id, float)
    lnp = ll - logsumexp(ll)          # defensive re-normalization
    return float(-np.sum(np.exp(lnp) * lnp))

# BASE[(ds, name)] = {"s": scores, "y": labels, "auroc": point estimate}
BASE = {}
for ds in DATASETS:
    with open(os.path.join(DATA_DIR, f"selfcheck_c2_{ds}_8b_results.json")) as f:
        d = json.load(f)
    det = [r for r in d["details"] if r.get("error") is None]
    BASE[(ds, "SelfCheckGPT")] = {
        "s": np.array([r["score"] for r in det], float),
        "y": np.array([int(r["is_halluc"]) for r in det], int),
        "auroc": d["auroc"]}

    with open(os.path.join(DATA_DIR, f"llm-check/llmcheck_c2_{ds}_8b_results.json")) as f:
        d = json.load(f)
    det = [r for r in d["details"] if r.get("error") is None]
    BASE[(ds, "LLM-Check")] = {
        "s": np.array([r["scores"]["logit_entropy"] for r in det], float),
        "y": np.array([int(r["is_halluc"]) for r in det], int),
        "auroc": d["auroc"]["logit_entropy"]}

    with open(os.path.join(DATA_DIR, REG[(ds, "8b", "bb")]["C2"])) as f:
        d = json.load(f)
    det = d["details"]
    s = np.array([semantic_entropy(r["log_lik_per_sem_id"]) for r in det], float)
    y = np.array([0 if r["judge_correct"] else 1 for r in det], int)
    BASE[(ds, "SemanticEntropy")] = {"s": s, "y": y, "auroc": fast_auroc(y, s)}

print("Semantic Entropy AUROC:",
      {DS_LABEL[ds]: round(BASE[(ds, 'SemanticEntropy')]['auroc'], 3) for ds in DATASETS})

In [ ]:
# Chart 6 — plain bars
SERIES67 = {
    "bb": [("C2", "C2 (holistic)", ""), ("C4:max", "C4 (max)", ""),
           ("C4:mean", "C4 (mean)", "//"),
           ("SelfCheckGPT", "SelfCheckGPT-NLI", "")],
    "wb": [("C2", "C2 (holistic)", ""), ("C4:max", "C4 (max)", ""),
           ("C4:mean", "C4 (mean)", "//"), ("C4:attn_mean", "C4 (attn_mean)", "xx"),
           ("LLM-Check", "LLM-Check (logit_entropy)", ""),
           ("SemanticEntropy", "Semantic Entropy", "")],
}

def auroc_of(ds, box, cond):
    if cond in BCOL:
        return BASE[(ds, cond)]["auroc"]
    return AUROC[(ds, "8b", box, cond)]

def color_of(cond):
    return BCOL.get(cond, COL.get(cond.split(":")[0]))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), sharey=True)
for ax, box in zip(axes, ["bb", "wb"]):
    series = SERIES67[box]
    k = len(series)
    w = 0.8 / k
    for j, (cond, label, hatch) in enumerate(series):
        xs = [i + (j - (k - 1) / 2) * w for i in range(len(DATASETS))]
        ys = [auroc_of(ds, box, cond) for ds in DATASETS]
        ax.bar(xs, ys, width=w * 0.92, color=color_of(cond),
               edgecolor="black", linewidth=0.5, hatch=hatch, label=label)
        for x, v in zip(xs, ys):
            ax.annotate(f"{v:.2f}", (x, v), textcoords="offset points",
                        xytext=(0, 2), ha="center", fontsize=6.5)
    ax.axhline(0.5, color=CHANCE, ls="--", lw=1, label="chance")
    ax.set_xticks(range(len(DATASETS)))
    ax.set_xticklabels([DS_LABEL[d] for d in DATASETS])
    ax.set_ylim(0, 1.0)
    ax.set_title("Black-box" if box == "bb" else "White-box")
    ax.legend(loc="upper right", fontsize=7.5, ncol=2)  # per-panel: series differ
axes[0].set_ylabel("AUROC")
fig.suptitle("C2 vs C4 vs external baselines (Llama-3.1-8B, N=1000)", y=1.02)
plt.tight_layout()
fig.savefig(f"{FIG_DIR}/cmp6_c2_c4_baselines.png", bbox_inches="tight")
plt.show()

**Notes — Chart 6**

- Purpose: position C2/C4 against established detectors under the *same* CoT
  setting. Baselines score the C2 chains; C4 scores its own stepwise chains.
- **Black-box:** SelfCheckGPT-NLI is strong on single-hop (TriviaQA 0.79,
  clearly above C2's 0.71) but **collapses to chance on MoreHopQA (0.50)** —
  sentence-vs-samples contradiction checking breaks when everything is long
  reasoning text. C4 `mean` (0.71) is the only black-box method here that
  stays useful on multi-hop.
- **White-box:** LLM-Check (`logit_entropy`) is flat across datasets
  (0.67–0.71) — remarkably hop-robust for a single-forward-pass method — and
  beats C2 everywhere. C4 beats or matches it on TriviaQA/NQ-Open; on
  MoreHopQA they are close (0.68 vs 0.71).
- Compare the printed Semantic Entropy values against C2: both use the same
  C2 samples and clustering, SE just replaces the kernel-entropy score with
  cluster-probability entropy — the gap between them is a direct measure of
  what the KLE kernel adds (or costs) on holistic chains.
- Fairness note for the meeting: LLM-Check and Semantic Entropy need logits /
  likelihoods (white-box); SelfCheckGPT and C2/C4-KLE black-box variants need
  only sampled text. RQ2 hinges on exactly this split.

In [ ]:
# Chart 7 — bootstrap version of Chart 6
CI7 = {}
for ds in DATASETS:
    for name in ["SelfCheckGPT", "LLM-Check", "SemanticEntropy"]:
        b = BASE[(ds, name)]
        lo, hi = bootstrap_ci(b["y"], b["s"])
        CI7[(ds, name)] = (fast_auroc(b["y"], b["s"]), lo, hi)

MARKER67 = {"C2": "o", "C4:max": "o", "C4:mean": "s", "C4:attn_mean": "^",
            "SelfCheckGPT": "o", "LLM-Check": "o", "SemanticEntropy": "D"}

def ci_of(ds, box, cond):
    return CI7[(ds, cond)] if cond in BCOL else CI[(ds, box, cond)]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), sharey=True)
for ax, box in zip(axes, ["bb", "wb"]):
    series = SERIES67[box]
    k = len(series)
    w = 0.8 / k
    for j, (cond, label, _) in enumerate(series):
        xs = [i + (j - (k - 1) / 2) * w for i in range(len(DATASETS))]
        vals = [ci_of(ds, box, cond) for ds in DATASETS]
        ys = [v[0] for v in vals]
        yerr = [[v[0] - v[1] for v in vals], [v[2] - v[0] for v in vals]]
        ax.errorbar(xs, ys, yerr=yerr, fmt=MARKER67[cond], ms=6, capsize=3,
                    lw=1.5, ls="none", color=color_of(cond), label=label)
        for x, v in zip(xs, ys):
            ax.annotate(f"{v:.2f}", (x, v), textcoords="offset points",
                        xytext=(5, -3), ha="left", fontsize=6.5)
    ax.axhline(0.5, color=CHANCE, ls="--", lw=1, label="chance")
    ax.set_xticks(range(len(DATASETS)))
    ax.set_xticklabels([DS_LABEL[d] for d in DATASETS])
    ax.set_title("Black-box" if box == "bb" else "White-box")
    ax.legend(loc="lower left", fontsize=7.5, ncol=2)  # per-panel: series differ
axes[0].set_ylabel("AUROC (95% bootstrap CI)")
fig.suptitle("C2 vs C4 vs external baselines, with bootstrap CIs "
             "(Llama-3.1-8B, N=1000)", y=1.02)
plt.tight_layout()
fig.savefig(f"{FIG_DIR}/cmp7_c2_c4_baselines_ci.png", bbox_inches="tight")
plt.show()

**Notes — Chart 7**

- Same comparison as Chart 6 with 95% bootstrap CIs (B=2000, question-level
  resampling; baseline CIs are computed in this cell, KLE CIs come from the
  Chart 3 cache).
- **SelfCheckGPT's MoreHopQA collapse is statistically solid:** its CI straddles
  0.5, i.e. indistinguishable from a random detector, while C4 `mean` sits far
  above with a non-overlapping interval.
- **On TriviaQA black-box, SelfCheckGPT (0.79) vs C2 (0.71):** intervals are
  clearly apart — the holistic KLE variant genuinely trails it there. This is
  an honest negative for C2, worth stating plainly; the black-box story is
  carried by C3/C4, not C2.
- **White-box MoreHopQA:** C4, LLM-Check and (check the printed value) Semantic
  Entropy CIs largely overlap — no single white-box winner on multi-hop; the
  RQ2-relevant point is that black-box C4 `mean` reaches the same level without
  internals.
- Remember these are unpaired per-run intervals: methods sharing the C2
  generations (baselines vs C2) can be compared more sharply with the paired
  bootstrap in the full-analysis notebook, section 13.

## Charts 8–9 — AUROC vs hallucination rate per condition

One panel per dataset; for every condition two adjacent bars: **AUROC**
(detector quality, blue) and **hallucination rate** (how often that condition's
own generations are judged wrong, gray). Both are proportions in [0, 1], so a
single shared axis is legitimate. The hallucination rate comes from the run
file, so C4's aggregation variants repeat the same gray bar by construction.
Llama-3.1-8B only. Chart 8 = white-box, Chart 9 = black-box.

In [ ]:
def auroc_vs_hrate(box, fname, title):
    conds = CONDS[box]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), sharey=True)
    for ax, ds in zip(axes, DATASETS):
        au = [AUROC[(ds, "8b", box, c)] for c in conds]
        hr = [HRATE[(ds, "8b", box, c.split(":")[0])] for c in conds]
        w = 0.38
        ax.bar([x - w / 2 for x in range(len(conds))], au, width=w * 0.92,
               color="#2a78d6", edgecolor="black", linewidth=0.5, label="AUROC")
        ax.bar([x + w / 2 for x in range(len(conds))], hr, width=w * 0.92,
               color="#9a9992", edgecolor="black", linewidth=0.5,
               label="hallucination rate")
        for x, (a, h) in enumerate(zip(au, hr)):
            ax.annotate(f"{a:.2f}", (x - w / 2, a), textcoords="offset points",
                        xytext=(0, 2), ha="center", fontsize=6.5)
            ax.annotate(f"{h:.2f}", (x + w / 2, h), textcoords="offset points",
                        xytext=(0, 2), ha="center", fontsize=6.5)
        ax.axhline(0.5, color=CHANCE, ls="--", lw=1, label="chance (AUROC)")
        ax.set_xticks(range(len(conds)))
        ax.set_xticklabels([c.replace(":", "\n") for c in conds], fontsize=9)
        ax.set_ylim(0, 1.08)
        ax.set_title(DS_LABEL[ds])
    axes[0].set_ylabel("proportion")
    axes[2].legend(loc="upper right", fontsize=8)
    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    fig.savefig(f"{FIG_DIR}/{fname}", bbox_inches="tight")
    plt.show()

# Chart 8 — white-box
auroc_vs_hrate("wb", "cmp8_wb_auroc_vs_hallucrate.png",
               "White-box: AUROC vs hallucination rate per condition "
               "(Llama-3.1-8B, N=1000)")

**Notes — Chart 8**

- Reading: the two bars answer different questions — gray: *how hard is the
  task under this condition's prompting* (base rate of judged-wrong answers);
  blue: *how well does the uncertainty score rank wrong above right*. They are
  deliberately side by side to show they are independent axes of quality.
- **C2 has the lowest hallucination rate on every dataset** (TriviaQA 0.19,
  NQ-Open 0.31, MoreHopQA 0.73): holistic CoT prompting genuinely improves
  answer accuracy. Yet its detection AUROC is among the weakest — *better
  generations ≠ better detectability*. Nice paradox for the meeting.
- **C3/C4 hallucination rates are higher than C2's** partly by construction:
  answer-span extraction forces a committed short answer, removing hedged
  responses the judge might otherwise accept.
- **MoreHopQA is in a different regime:** 73–85% hallucination rate. With such
  label imbalance AUROC remains well-defined, but threshold metrics
  (accuracy/F1 of a deployed detector) would be dominated by the positive
  class — worth saying explicitly when presenting.
- Within a dataset the gray bars barely move across C1/C3/C4 while the blue
  ones swing (e.g. MoreHopQA WB: rates 0.81–0.85 but AUROC 0.47 → 0.68): AUROC
  differences reflect the *scoring method*, not shifts in task difficulty.

In [ ]:
# Chart 9 — black-box
auroc_vs_hrate("bb", "cmp9_bb_auroc_vs_hallucrate.png",
               "Black-box: AUROC vs hallucination rate per condition "
               "(Llama-3.1-8B, N=1000)")

**Notes — Chart 9**

- Same reading as Chart 8, black-box scores. The gray bars are nearly identical
  to Chart 8's (each box type has its own generations, but rates agree within
  ±0.02, e.g. TriviaQA C1 0.244 vs 0.243) — a useful sanity check that the
  BB/WB pipelines sample from the same distribution.
- **The C1-MoreHopQA pathology in full view:** 82% of direct answers are wrong,
  and AUROC 0.37 is *below* the chance line — on multi-hop the baseline's
  uncertainty score ranks wrong answers as *more* confident. Both facts argue
  for the CoT conditions.
- **C2's trade-off is sharpest here:** black-box MoreHopQA — best generation
  quality of all conditions (0.71 rate vs 0.81–0.85) but near-chance detection
  (0.55). If one had to deploy a single condition, this chart shows why the
  choice is genuinely two-dimensional.
- **C4 on MoreHopQA:** highest hallucination rate (0.85) *and* the best
  black-box detection (`mean` 0.71) — the stepwise score works exactly where
  the task is hardest, which is the thesis pitch for RQ1 in one panel.
- Single-hop black-box remains C3 territory (TriviaQA 0.86 / NQ-Open 0.74,
  shown in Chart 1); this pair of charts adds that those wins do not come from
  easier base rates.

## Charts 10–11 — token cost and the performance-cost trade-off

Token accounting per question (mean over N=1000): **generated tokens**
(sampling: 10 high-temp samples + low-temp candidate; for C4 also the step
continuations) plus, where present, **forward-pass tokens** (white-box hidden
state / attention extraction). Prompt tokens are negligible and excluded.

Two deliberate deviations from the original trade-off cell in
`kle_cot_triviaqa_analysis.ipynb`:
- **No wall-clock axis.** C4 generates with batched `num_return_sequences`
  while C1–C3 loop sequentially, so seconds are not comparable across
  conditions — token counts are the fair cost metric (cross-check with Slurm
  `sacct` is planned separately).
- **No dual y-axis.** AUROC and tokens have different units, so Chart 10 uses
  two rows sharing the x-axis instead of twin axes.

C4 is represented by one AUROC per box (black-box `mean`, white-box
`attn_mean`); its token cost is identical across aggregations anyway.

In [ ]:
# Mean tokens per question, split into generation vs forward-pass components:
# TOK_GEN / TOK_FWD, with TOKENS = their sum (used by Charts 10-11)
TOK_GEN, TOK_FWD, TOKENS = {}, {}, {}
for ds in DATASETS:
    for box in ["bb", "wb"]:
        for cond in ["C1", "C2", "C3", "C4"]:
            with open(os.path.join(DATA_DIR, REG[(ds, "8b", box)][cond])) as f:
                d = json.load(f)
            gen_pq, fwd_pq = [], []
            for r in d["details"]:
                c = r.get("cost") or {}
                gen_pq.append(c.get("n_gen_tokens", c.get("gen_tokens", 0)))
                fwd_pq.append(c.get("n_fwd_tokens", c.get("fwd_tokens", 0)))
            TOK_GEN[(ds, box, cond)] = sum(gen_pq) / len(gen_pq)
            TOK_FWD[(ds, box, cond)] = sum(fwd_pq) / len(fwd_pq)
            TOKENS[(ds, box, cond)] = TOK_GEN[(ds, box, cond)] + TOK_FWD[(ds, box, cond)]

def rep_auroc(ds, box, cond):
    """One AUROC per condition: C4 -> mean (BB) / attn_mean (WB)."""
    if cond == "C4":
        cond = "C4:mean" if box == "bb" else "C4:attn_mean"
    return AUROC[(ds, "8b", box, cond)]

def fmt_tok(v):
    return f"{v / 1000:.1f}K" if v >= 1000 else f"{v:.0f}"

print({(DS_LABEL[ds], box): fmt_tok(TOKENS[(ds, box, "C4")])
       for ds in DATASETS for box in ["bb", "wb"]})

In [ ]:
# Chart 10 — AUROC (top row) vs mean tokens/question (bottom row)
BOX_COL = {"bb": "#2a78d6", "wb": "#eb6834"}
BOX_LABEL = {"bb": "black-box", "wb": "white-box"}
CB = ["C1", "C2", "C3", "C4"]

fig, axes = plt.subplots(2, 3, figsize=(15, 7.6), sharex="col")
for col, ds in enumerate(DATASETS):
    ax_a, ax_t = axes[0][col], axes[1][col]
    w = 0.38
    for j, box in enumerate(["bb", "wb"]):
        xs = [i + (j - 0.5) * w for i in range(len(CB))]
        au = [rep_auroc(ds, box, c) for c in CB]
        tk = [TOKENS[(ds, box, c)] for c in CB]
        ax_a.bar(xs, au, width=w * 0.92, color=BOX_COL[box],
                 edgecolor="black", linewidth=0.5, label=BOX_LABEL[box])
        ax_t.bar(xs, tk, width=w * 0.92, color=BOX_COL[box],
                 edgecolor="black", linewidth=0.5, label=BOX_LABEL[box])
        for x, v in zip(xs, au):
            ax_a.annotate(f"{v:.2f}", (x, v), textcoords="offset points",
                          xytext=(0, 2), ha="center", fontsize=6.5)
        for x, v in zip(xs, tk):
            ax_t.annotate(fmt_tok(v), (x, v), textcoords="offset points",
                          xytext=(0, 2), ha="center", fontsize=6.5)
    ax_a.axhline(0.5, color=CHANCE, ls="--", lw=1)
    ax_a.set_ylim(0, 1.0)
    ax_a.set_title(DS_LABEL[ds])
    ax_t.set_xticks(range(len(CB)))
    ax_t.set_xticklabels(["C1", "C2", "C3", "C4*"])
axes[0][0].set_ylabel("AUROC")
axes[1][0].set_ylabel("mean tokens / question")
axes[0][2].legend(loc="upper right", fontsize=8)
fig.suptitle("AUROC (top) vs token cost (bottom) per condition "
             "(Llama-3.1-8B, N=1000; C4* = mean/attn_mean representative)", y=1.0)
plt.tight_layout()
fig.savefig(f"{FIG_DIR}/cmp10_auroc_vs_tokens.png", bbox_inches="tight")
plt.show()

**Notes — Chart 10**

- Reading: top = detection quality, bottom = what it costs in tokens; columns
  share the x-axis, so scan vertically per condition.
- **The cost ladder is roughly C1 << C3 ≤ C2 << C4** on every dataset: CoT
  conditions pay ~5–10× more generated tokens than the direct-answer baseline,
  and C4's step continuations put it on top by a clear margin.
- **C3 is slightly cheaper than C2** despite being the stronger detector on
  single-hop — extraction does not add generation cost (same chains, plus a
  cheap span step); there is no cost argument for preferring C2.
- **Black-box vs white-box bars differ mainly through forward-pass tokens:**
  generation cost is essentially the same per condition; the white-box surplus
  (hidden-state/attention extraction, largest for C4) is forward-pass load,
  which is much cheaper per token than autoregressive generation — worth saying
  when someone reads the bottom row as "white-box is several times slower".
- On MoreHopQA everything roughly doubles (longer chains, more steps at
  `max_steps=5`) — cost scales with hop count, which matters for the "is C4
  worth it" question answered in Chart 11.

In [ ]:
# Chart 11 — performance-cost trade-off (token-based), one row per box type;
# bubble size = hallucination rate of that condition's own generations
from matplotlib.lines import Line2D

SIZE_SCALE = 400  # bubble area = hallucination rate * SIZE_SCALE (area-true)

fig, axes = plt.subplots(2, 3, figsize=(15, 8.4), sharey=True)
for row, box in enumerate(["bb", "wb"]):
    for col, ds in enumerate(DATASETS):
        ax = axes[row][col]
        for c in CB:
            x = TOKENS[(ds, box, c)]
            y = rep_auroc(ds, box, c)
            hr = HRATE[(ds, "8b", box, c)]
            ax.scatter(x, y, marker="o", s=hr * SIZE_SCALE, color=COL[c],
                       alpha=0.85, edgecolor="black", linewidth=0.6, zorder=3)
            ax.annotate(c, (x, y), textcoords="offset points",
                        xytext=(7, 7), fontsize=7.5)
        ax.axhline(0.5, color=CHANCE, ls="--", lw=1)
        ax.set_xscale("log")
        ax.set_title(f"{DS_LABEL[ds]} — {BOX_LABEL[box]}")
        if row == 1:
            ax.set_xlabel("mean tokens / question (log)")
    axes[row][0].set_ylabel("AUROC")
cond_handles = [Line2D([0], [0], marker="o", ls="none", color=COL[c], label=c)
                for c in CB]
size_handles = [Line2D([0], [0], marker="o", ls="none", mfc="none", mec="black",
                       ms=(r * SIZE_SCALE) ** 0.5, label=f"{r:.0%} halluc. rate")
                for r in [0.2, 0.5, 0.8]]
axes[0][2].legend(handles=cond_handles, loc="lower right", fontsize=8)
axes[1][2].legend(handles=size_handles, loc="lower right", fontsize=7.5,
                  labelspacing=1.1, borderpad=0.9)
fig.suptitle("Performance-cost trade-off, token-based; bubble size = hallucination "
             "rate (Llama-3.1-8B, N=1000; C4 = mean/attn_mean representative)", y=1.0)
plt.tight_layout()
fig.savefig(f"{FIG_DIR}/cmp11_tradeoff_tokens.png", bbox_inches="tight")
plt.show()

**Notes — Chart 11**

- Reading: up = better detection, right = more expensive; the interesting
  methods are the ones no other point sits both above and to the left of
  (the efficiency frontier). Top row = black-box, bottom row = white-box;
  color = condition; **bubble area = hallucination rate** of that condition's
  own generations (size legend bottom-right).
- The third dimension adds the generation-quality context: TriviaQA bubbles are
  small (~0.19–0.27 rate), MoreHopQA bubbles are large (~0.71–0.85) — the same
  detector operates in very different regimes. Within every panel C2 is the
  smallest bubble (holistic CoT produces the fewest hallucinations) while
  sitting low on AUROC: generation quality and detectability pull apart again.
- **Single-hop (TriviaQA/NQ-Open): C1 is the efficiency king** — an order of
  magnitude cheaper and within a few AUROC points of the best. C3 buys the top
  detection score for ~5–10× the tokens; C2 is dominated by C3 (more or equal
  cost, less AUROC) and has no place on the frontier.
- **Multi-hop (MoreHopQA) reshuffles the frontier:** cheap C1 is now *below
  chance* — its low cost buys nothing — and black-box C4 `mean`, the most
  expensive point, is the only black-box method clearly worth its tokens.
  Cost-effectiveness of stepwise detection is hop-dependent, which is the
  compute-side complement of RQ1.
- Comparing rows: each white-box point sits right of its black-box twin
  (forward-pass surplus) with mixed AUROC effects — paying the white-box
  overhead is only justified for C4 on single-hop, echoing Chart 4.
- Caveat to repeat in the meeting: token counts, not seconds (batching makes
  wall-clock incomparable across conditions); and the C4 point uses one
  representative aggregation per box.

## Chart 12 — token cost decomposed: generation vs forward-pass

Chart 10's layout with the bottom row **stacked**: solid = generated tokens
(autoregressive sampling — the expensive kind), hatched = forward-pass tokens
(teacher-forced re-encoding for hidden states / attention — parallel over the
sequence, much cheaper per token). Black-box bars have no hatched segment by
construction. This answers "*why is white-box C4 so expensive*" in one look.

In [ ]:
from matplotlib.patches import Patch

fig, axes = plt.subplots(2, 3, figsize=(15, 7.6), sharex="col")
for col, ds in enumerate(DATASETS):
    ax_a, ax_t = axes[0][col], axes[1][col]
    w = 0.38
    for j, box in enumerate(["bb", "wb"]):
        xs = [i + (j - 0.5) * w for i in range(len(CB))]
        au = [rep_auroc(ds, box, c) for c in CB]
        gen = [TOK_GEN[(ds, box, c)] for c in CB]
        fwd = [TOK_FWD[(ds, box, c)] for c in CB]
        ax_a.bar(xs, au, width=w * 0.92, color=BOX_COL[box],
                 edgecolor="black", linewidth=0.5, label=BOX_LABEL[box])
        ax_t.bar(xs, gen, width=w * 0.92, color=BOX_COL[box],
                 edgecolor="black", linewidth=0.5)
        ax_t.bar(xs, fwd, width=w * 0.92, bottom=gen, color=BOX_COL[box],
                 alpha=0.45, hatch="//", edgecolor="black", linewidth=0.5)
        for x, v in zip(xs, au):
            ax_a.annotate(f"{v:.2f}", (x, v), textcoords="offset points",
                          xytext=(0, 2), ha="center", fontsize=6.5)
        for x, g, f_ in zip(xs, gen, fwd):
            ax_t.annotate(fmt_tok(g + f_), (x, g + f_), textcoords="offset points",
                          xytext=(0, 2), ha="center", fontsize=6.5)
    ax_a.axhline(0.5, color=CHANCE, ls="--", lw=1)
    ax_a.set_ylim(0, 1.0)
    ax_a.set_title(DS_LABEL[ds])
    ax_t.set_xticks(range(len(CB)))
    ax_t.set_xticklabels(["C1", "C2", "C3", "C4*"])
axes[0][0].set_ylabel("AUROC")
axes[1][0].set_ylabel("mean tokens / question")
axes[0][2].legend(loc="upper right", fontsize=8)
axes[1][2].legend(handles=[
    Patch(facecolor=BOX_COL["bb"], edgecolor="black", label="black-box (generation only)"),
    Patch(facecolor=BOX_COL["wb"], edgecolor="black", label="white-box: generation"),
    Patch(facecolor=BOX_COL["wb"], edgecolor="black", alpha=0.45, hatch="//",
          label="white-box: forward-pass"),
], loc="upper left", fontsize=7.5)
fig.suptitle("AUROC (top) vs token cost decomposed into generation + forward-pass "
             "(bottom) (Llama-3.1-8B, N=1000; C4* = mean/attn_mean representative)", y=1.0)
plt.tight_layout()
fig.savefig(f"{FIG_DIR}/cmp12_auroc_vs_tokens_stacked.png", bbox_inches="tight")
plt.show()

**Notes — Chart 12**

- The solid segments (generation) are nearly identical between the black-box
  and white-box bars of the same condition — same sampling configuration,
  independent runs. Everything the white-box bars add on top is hatched
  forward-pass load.
- **Forward-pass dominates every white-box stack** (roughly 60–85% of the
  total): every sample is re-encoded together with its full context (prompt +
  chain prefix), and for C4 this happens per step with a *growing* prefix —
  which is exactly why white-box C4 towers over everything, worst on MoreHopQA
  (`max_steps=5`, long chains).
- **Do not read stack height as runtime.** Hatched tokens are single parallel
  forward passes; solid tokens are sequential autoregressive decoding. In
  wall-clock terms the solid segment is far more expensive per token — the
  stack shows *token accounting*, not speed.
- The C4 generation segment alone (step-wise resampling: 10 continuations per
  step) is already ~2–5× the C2/C3 generation cost; the stepwise method is
  intrinsically more expensive even before any white-box instrumentation.
- Engineering headroom worth one sentence in the thesis: shared-prefix KV
  caching would cut most of the hatched segment; the current pipeline
  re-encodes every continuation independently, so these forward numbers are an
  upper bound.

## Charts 13–18 — additional comparison metrics: AUPRC, PRR, FPR@95TPR

Three metrics that complement AUROC, each as its own figure in the Chart 3
layout (dot + 95% bootstrap CI, B=2000). Charts 13–15 compare C1–C4;
Charts 16–18 compare C2/C4 against the external baselines (Chart 6/7 scope).

- **AUPRC** (average precision): area under the precision-recall curve for the
  hallucination class. Unlike AUROC it depends on the positive rate — the
  chance level is the prevalence itself, drawn as a short dashed segment per
  dataset. Read it as *lift over prevalence*, and never compare raw AUPRC
  across conditions/datasets with different rates.
- **PRR** (prediction rejection ratio): how much of the ideal rejection gain
  the score achieves, normalized between random rejection (0) and an oracle
  that knows the labels (1). Unlike raw AUARC it *is* comparable across
  conditions with different base accuracies. Negative = worse than random.
- **FPR@95TPR**: the fraction of *correct* answers falsely flagged when the
  threshold is set to catch 95% of hallucinations. **Lower is better**; a
  random detector sits at 0.95.

Runtime note: each figure bootstraps its runs on first execution (~1–2 min per
figure); results are cached in `MCI`. Requires the Chart 3 helpers and (for
Charts 16–18) the Chart 6 baseline cell — run top to bottom.

In [ ]:
from sklearn.metrics import average_precision_score, roc_curve

def auprc(y, s):
    return average_precision_score(y, s)

def fpr_at_95tpr(y, s):
    fpr, tpr, _ = roc_curve(y, s)
    return float(fpr[np.argmax(tpr >= 0.95)])

def prr(y, s, seed=RNG_SEED):
    """Prediction rejection ratio: 0 = random rejection, 1 = oracle."""
    rng = np.random.default_rng(seed)
    n = len(y)
    order = np.lexsort((rng.random(n), s))       # ascending s = most confident first
    ks = np.arange(1, n + 1)
    auarc = (np.cumsum((1 - y)[order]) / ks).mean()
    base = (1 - y).mean()
    oracle = (np.cumsum(np.sort(1 - y)[::-1]) / ks).mean()
    return (auarc - base) / (oracle - base)

METRICS = {"AUPRC": auprc, "PRR": prr, "FPR@95TPR": fpr_at_95tpr}

def bootstrap_metric_ci(y, s, fn, n_boot=N_BOOT, seed=RNG_SEED):
    rng = np.random.default_rng(seed)
    n = len(y); out = np.full(n_boot, np.nan)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        yb = y[idx]
        if 0 < yb.sum() < n:
            out[b] = fn(yb, s[idx])
    return np.nanpercentile(out, [2.5, 97.5])

# Raw per-question scores for all KLE runs (baselines already sit in BASE)
SY = {}
for ds in DATASETS:
    for box in ["bb", "wb"]:
        cache = {}
        for cond, _, _ in BARS[box]:
            base_c = cond.split(":")[0]
            if base_c not in cache:
                with open(os.path.join(DATA_DIR, REG[(ds, "8b", box)][base_c])) as f:
                    cache[base_c] = json.load(f)
            SY[(ds, box, cond)] = extract_scores(cache[base_c], box, cond)

def sy_of(ds, box, cond):
    if cond in BCOL:
        b = BASE[(ds, cond)]
        return b["s"], b["y"]
    return SY[(ds, box, cond)]

MCI = {}  # (metric, ds, box, cond) -> (value, lo, hi), lazily filled
def metric_ci(metric, ds, box, cond):
    key = (metric, ds, box, cond)
    if key not in MCI:
        s, y = sy_of(ds, box, cond)
        fn = METRICS[metric]
        lo, hi = bootstrap_metric_ci(y, s, fn)
        MCI[key] = (fn(y, s), lo, hi)
    return MCI[key]

def metric_dot_chart(metric, scope, fname):
    """scope: 'conds' (C1-C4, Chart 3 series) or 'baselines' (Chart 6/7 series)."""
    series_map = BARS if scope == "conds" else SERIES67
    marker_map = MARKER if scope == "conds" else MARKER67
    fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8), sharey=True)
    for ax, box in zip(axes, ["bb", "wb"]):
        series = series_map[box]
        k = len(series)
        w = 0.8 / k
        for j, (cond, label, _) in enumerate(series):
            xs = [i + (j - (k - 1) / 2) * w for i in range(len(DATASETS))]
            vals = [metric_ci(metric, ds, box, cond) for ds in DATASETS]
            ys = [v[0] for v in vals]
            yerr = [[v[0] - v[1] for v in vals], [v[2] - v[0] for v in vals]]
            ax.errorbar(xs, ys, yerr=yerr, fmt=marker_map[cond], ms=6, capsize=3,
                        lw=1.5, ls="none", color=color_of(cond), label=label)
            for x, v in zip(xs, ys):
                ax.annotate(f"{v:.2f}", (x, v), textcoords="offset points",
                            xytext=(5, -3), ha="left", fontsize=6.5)
        if metric == "PRR":
            ax.axhline(0, color=CHANCE, ls="--", lw=1, label="random")
        elif metric == "FPR@95TPR":
            ax.axhline(0.95, color=CHANCE, ls="--", lw=1, label="random")
        else:  # AUPRC: chance level = prevalence, per dataset (mean over series)
            for i, ds in enumerate(DATASETS):
                prev = np.mean([sy_of(ds, box, c)[1].mean() for c, _, _ in series])
                ax.hlines(prev, i - 0.4, i + 0.4, color=CHANCE, ls="--", lw=1,
                          label="positive rate" if i == 0 else None)
        ax.set_xticks(range(len(DATASETS)))
        ax.set_xticklabels([DS_LABEL[d] for d in DATASETS])
        ax.set_title("Black-box" if box == "bb" else "White-box")
        ax.legend(loc="best", fontsize=7, ncol=2)
    better = " (lower is better)" if metric == "FPR@95TPR" else ""
    axes[0].set_ylabel(f"{metric}{better}, 95% bootstrap CI")
    scope_txt = ("by condition" if scope == "conds"
                 else "C2/C4 vs external baselines")
    fig.suptitle(f"{metric} {scope_txt} (Llama-3.1-8B, N=1000)", y=1.02)
    plt.tight_layout()
    fig.savefig(f"{FIG_DIR}/{fname}", bbox_inches="tight")
    plt.show()

print("metric helpers ready")

In [ ]:
# Chart 13 — AUPRC, C1-C4
metric_dot_chart("AUPRC", "conds", "cmp13_auprc_conditions.png")

**Notes — Chart 13**

- AUPRC's chance level is the dashed *positive-rate* segment, not 0.5: on
  MoreHopQA (~0.8 prevalence) even a random detector scores ~0.8, so the
  absolute numbers there are inflated by construction. Always read the gap
  between a point and its dataset's dashed segment (the *lift*).
- Because prevalence differs per dataset (and slightly per condition), do not
  rank datasets or conditions by raw AUPRC — within-dataset ordering is the
  only safe comparison.
- Expectation from the AUROC results (verify against the plot): C3 should show
  the largest lift on single-hop, C4 `mean` on MoreHopQA black-box, and
  C1-MoreHopQA should sit *below* its dashed segment — the precision-recall
  view of the inverted signal.
- AUPRC weights early precision: it punishes detectors whose top-ranked alarms
  are wrong, which threshold-free AUROC partially hides — useful if the
  deployment story is "review the top-k flagged answers".

In [ ]:
# Chart 14 — PRR, C1-C4
metric_dot_chart("PRR", "conds", "cmp14_prr_conditions.png")

**Notes — Chart 14**

- PRR rescales the rejection benefit between random (0) and oracle (1), which
  removes the base-accuracy confound — unlike raw AUARC, these values *are*
  comparable across conditions and datasets. This is the fairest single number
  in the notebook for "how useful is this score in practice".
- Negative PRR means rejecting by this score is *worse* than rejecting at
  random — expect C1 on MoreHopQA to land there (the selective-prediction
  reading of AUROC < 0.5).
- PRR and AUROC usually agree on ordering but not always on magnitude: a score
  can rank well overall (decent AUROC) yet misplace its most-confident tail,
  which PRR punishes. Divergences between Chart 3 and this chart are worth a
  sentence in the thesis.
- Rough calibration for reading: PRR above ~0.5 is a strong practical
  detector; around 0.2–0.3 the rejection gain is real but modest.

In [ ]:
# Chart 15 — FPR@95TPR, C1-C4
metric_dot_chart("FPR@95TPR", "conds", "cmp15_fpr95_conditions.png")

**Notes — Chart 15**

- Operating-point metric, *lower is better*: with the threshold set to catch
  95% of hallucinations, this is the share of correct answers wrongly flagged.
  Random scoring sits at the dashed 0.95 line.
- This is the harshest of the three metrics — demanding 95% recall pushes the
  threshold deep into the score overlap, so values above ~0.8 (i.e. flagging
  almost everything) are common for mid-0.7 AUROC detectors. Expect only the
  strongest cells (e.g. C3 on TriviaQA) to look usable here.
- CIs are wider than for AUROC/AUPRC: the statistic depends on one tail of the
  score distribution, so single resampled questions move it — be extra careful
  about over-reading gaps.
- Framing for the meeting: AUROC says *whether* the signal exists, PRR says
  *how much* of it is usable, FPR@95TPR says *what it costs* to act on it at a
  strict recall target.

In [ ]:
# Chart 16 — AUPRC, C2/C4 vs baselines
metric_dot_chart("AUPRC", "baselines", "cmp16_auprc_baselines.png")

**Notes — Chart 16**

- Same reading as Chart 13; the baselines share the C2 generations, so within a
  dataset their prevalence is (nearly) identical and their AUPRC values are
  directly comparable with C2's — the fairest baseline-vs-C2 comparison in the
  metric set. C4's prevalence differs slightly (own generations).
- Check whether SelfCheckGPT's strong TriviaQA AUROC survives in PR view: a
  method can win AUROC yet lose early precision. Conversely on MoreHopQA its
  point should sit on the dashed prevalence segment (chance).
- Semantic Entropy vs C2 remains the cleanest ablation (same samples, same
  clustering, different score); if their AUPRC gap is larger than their AUROC
  gap, the kernel score is specifically mis-ranking the top of the list.

In [ ]:
# Chart 17 — PRR, C2/C4 vs baselines
metric_dot_chart("PRR", "baselines", "cmp17_prr_baselines.png")

**Notes — Chart 17**

- The deployment-oriented ranking of methods: which score, used as a rejection
  rule, recovers the largest share of the oracle's benefit.
- SelfCheckGPT on MoreHopQA should sit at ≈0 (random) — the PRR restatement of
  its collapse; C4 `mean` black-box is expected to keep a clearly positive PRR
  there, which is the strongest practical argument for stepwise KLE on
  multi-hop.
- Baselines and C2 share generations (same base accuracy), so their PRR
  differences reflect scoring quality alone; C4 rows additionally differ in
  base accuracy, which PRR's normalization absorbs — that is exactly why this
  metric was chosen.

In [ ]:
# Chart 18 — FPR@95TPR, C2/C4 vs baselines
metric_dot_chart("FPR@95TPR", "baselines", "cmp18_fpr95_baselines.png")

**Notes — Chart 18**

- Lower is better; dashed line = random (0.95). Expect most methods to look
  weak here — at 95% recall the strict threshold exposes every score's overlap
  region; differences near the line are not meaningful (wide CIs, tail-driven
  statistic).
- The interesting cells are the ones meaningfully *below* the line: they mark
  method×dataset combinations where a high-recall hallucination filter is
  actually deployable.
- If two methods tie on AUROC but split here (or in Chart 17), prefer the one
  with better tail behavior for the thesis's practical recommendation; settle
  borderline pairs with the paired bootstrap (full-analysis notebook,
  section 13) rather than CI overlap.